# 4. 神经形态系统脉冲编码（Vin 输入）

sEMG 幅值 → 三段线性电压映射（1.0～2.5 V）→ `System_with_TIA` 电路 → 二值脉冲。

In [ ]:
# 用户参数：远程全量运行通常只需修改本单元。
import os

T = 200
BACKEND = "cuda"  # cuda / auto / numpy；目标 RTX 5090 环境默认强制使用 CUDA。
QUANTILE_DEVICE = "cuda"  # cuda / auto / cpu；全局分位点默认也在 GPU 上计算。

P_LOW = 0.10
P_MID = 0.90
P_HIGH = 0.995

CUDA_BATCH_SAMPLES = 64
CPU_BATCH_SAMPLES = 8
IO_WORKERS = 8
CUDA_BENCHMARK_STREAMS = 8192
QUANTILE_HISTOGRAM_BINS = 1 << 20
OVERWRITE = False
REUSE_MAPPING_PARAMETERS = True
CUDA_BUILD_VERBOSE = False

if isinstance(T, bool) or not isinstance(T, int) or T <= 0:
    raise ValueError("T 必须是正整数。")
if BACKEND not in {"auto", "cuda", "numpy"}:
    raise ValueError("BACKEND 必须是 auto、cuda 或 numpy。")
if QUANTILE_DEVICE not in {"auto", "cuda", "cpu"}:
    raise ValueError("QUANTILE_DEVICE 必须是 auto、cuda 或 cpu。")
if not 0 < P_LOW < P_MID < P_HIGH < 1:
    raise ValueError("分位参数必须满足 0 < P_LOW < P_MID < P_HIGH < 1。")
if min(CUDA_BATCH_SAMPLES, CPU_BATCH_SAMPLES, IO_WORKERS) <= 0:
    raise ValueError("批大小和 IO_WORKERS 必须为正整数。")
if QUANTILE_HISTOGRAM_BINS <= 0:
    raise ValueError("QUANTILE_HISTOGRAM_BINS 必须为正整数。")

In [ ]:
import hashlib
import json
import math
import os
import platform
import sys
import time
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import torch
from tqdm.auto import tqdm


def find_project_root():
    # 兼容从仓库根目录、notebook 目录或远程 Jupyter 工作目录启动。
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        required = (
            candidate / "CapgMyo_data",
            candidate / "neurophic_system_model" / "system_with_tia.py",
            candidate / "neurophic_system_model" / "system_with_tia_batch.py",
            candidate / "src" / "data_prepare",
        )
        if all(path.exists() for path in required):
            return candidate
    raise RuntimeError("无法定位 CapgMyo 项目根目录。")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from neurophic_system_model.system_with_tia_batch import (
    INTERNAL_STEP_COUNT,
    INTERNAL_STEP_S,
    simulate_system_batch_spikes,
    simulate_system_batch_spikes_reference,
)

DATA_ROOT = PROJECT_ROOT / "CapgMyo_data"
OUTPUT_BASE = DATA_ROOT / "neurophic_system_encoding_spike"
OUTPUT_ROOT = OUTPUT_BASE / f"T_{T}"
MAPPING_PATH = OUTPUT_BASE / "mapping.json"
MANIFEST_PATH = OUTPUT_ROOT / "manifest.json"

SPLIT_NAMES = ("train", "val", "test")
EXPECTED_COUNTS = {"train": 1008, "val": 144, "test": 288}
# 只编码正负分离数据；映射统计量仍来自 raw/train 的绝对幅值。
SOURCE_CONFIGS = {
    "raw_polarity": {"root": DATA_ROOT / "raw_polarity", "streams": 2},
}
RAW_ROOT = DATA_ROOT / "raw"
RAW_TRAIN_ROOT = RAW_ROOT / "train"
RAW_MANIFEST_PATH = RAW_ROOT / "split_manifest.json"

cuda_runtime_available = torch.cuda.is_available()
cuda_toolkit_available = False
if cuda_runtime_available:
    from torch.utils.cpp_extension import CUDA_HOME
    cuda_toolkit_available = CUDA_HOME is not None

if BACKEND == "auto":
    RESOLVED_BACKEND = "cuda" if cuda_toolkit_available else "numpy"
else:
    RESOLVED_BACKEND = BACKEND
if RESOLVED_BACKEND == "cuda" and not cuda_runtime_available:
    raise RuntimeError("已指定 CUDA 后端，但 torch.cuda.is_available() 为 False。")
if RESOLVED_BACKEND == "cuda" and not cuda_toolkit_available:
    raise RuntimeError("CUDA 批量求解器需要 CUDA Toolkit/nvcc，但当前环境未检测到 CUDA_HOME。")

if QUANTILE_DEVICE == "auto":
    RESOLVED_QUANTILE_DEVICE = "cuda" if cuda_runtime_available else "cpu"
else:
    RESOLVED_QUANTILE_DEVICE = QUANTILE_DEVICE
if RESOLVED_QUANTILE_DEVICE == "cuda" and not cuda_runtime_available:
    raise RuntimeError("已指定使用 CUDA 计算分位点，但当前 PyTorch 未发现 CUDA 设备。")

CPU_THREADS = max(1, min(os.cpu_count() or 1, 25))
torch.set_num_threads(CPU_THREADS)

print(f"项目根目录：{PROJECT_ROOT}")
print(f"输出目录：{OUTPUT_ROOT}")
print(f"映射参数：{MAPPING_PATH}")
print(f"T={T}, 仿真时长=1 s, 内部步数={INTERNAL_STEP_COUNT}, 步长={INTERNAL_STEP_S:g} s")
print(f"求解后端：{RESOLVED_BACKEND}")
print(f"分位统计设备：{RESOLVED_QUANTILE_DEVICE}，CPU 线程：{CPU_THREADS}，I/O 线程：{IO_WORKERS}")
if cuda_runtime_available:
    print(f"CUDA：{torch.cuda.get_device_name(torch.cuda.current_device())}")
    print(f"显存：{torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GiB")

In [ ]:
V_ZERO = 1.0
V_ON = 1.8639
V_MID = 2.4
V_MAX = 2.5

REFERENCE_CHECK_CHANNELS = 4
MIN_EXACT_F1 = 0.95
MIN_TOLERANT_F1 = 0.995
MAX_SPIKE_COUNT_DIFFERENCE = 1


def load_pt(path):
    try:
        return torch.load(path, map_location="cpu", weights_only=True)
    except TypeError:
        return torch.load(path, map_location="cpu")


def save_pt_atomic(path, payload):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_suffix(path.suffix + ".tmp")
    torch.save(payload, temporary_path)
    os.replace(temporary_path, path)


def write_json_atomic(path, payload):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_suffix(path.suffix + ".tmp")
    temporary_path.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    os.replace(temporary_path, path)


def sha256sum(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def canonical_hash(payload):
    serialized = json.dumps(
        payload,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    ).encode("utf-8")
    return hashlib.sha256(serialized).hexdigest()


def validate_source_payload(payload, path, streams):
    if not isinstance(payload, dict) or "data" not in payload:
        raise TypeError(f"{path} 必须是包含 data 的字典。")
    data = payload["data"]
    expected_shape = (1000, streams, 8, 16)
    if not isinstance(data, torch.Tensor):
        raise TypeError(f"{path} 的 data 必须是 Tensor。")
    if tuple(data.shape) != expected_shape or data.dtype != torch.float32:
        raise ValueError(
            f"{path} data={tuple(data.shape)}/{data.dtype}，"
            f"期望 {expected_shape}/torch.float32。"
        )
    if not torch.isfinite(data).all():
        raise ValueError(f"{path} 包含 NaN 或无穷大。")
    if streams == 2 and torch.any(data < 0):
        raise ValueError(f"{path} 的 polarity 幅值不应包含负数。")
    for key in ("label", "subject_id", "gesture_id", "repetition_id"):
        if key not in payload:
            raise KeyError(f"{path} 缺少元数据字段 {key}。")
    return data


def validate_mapping_parameters(mapping):
    probabilities = mapping["probabilities"]
    anchors = mapping["voltage_anchors_v"]
    q_low = float(mapping["q_low"])
    q_mid = float(mapping["q_mid"])
    q_high = float(mapping["q_high"])
    if probabilities != {"low": P_LOW, "mid": P_MID, "high": P_HIGH}:
        raise ValueError("映射文件中的分位概率与当前配置不一致。")
    if anchors != {"zero": V_ZERO, "on": V_ON, "mid": V_MID, "max": V_MAX}:
        raise ValueError("映射文件中的电压锚点与定义不一致。")
    if not np.all(np.isfinite([q_low, q_mid, q_high])):
        raise ValueError("实际分位点包含 NaN 或无穷大。")
    if not 0 < q_low < q_mid < q_high:
        raise ValueError("实际分位点必须满足 0 < q_low < q_mid < q_high。")
    return q_low, q_mid, q_high


def map_activity_vin(activity, mapping, statistics=None, prefix=None):
    q_low, q_mid, q_high = validate_mapping_parameters(mapping)
    activity = np.asarray(activity, dtype=np.float64)
    if not np.all(np.isfinite(activity)) or np.any(activity < 0):
        raise ValueError("Vin 映射输入必须是有限的非负活动量。")

    voltage = np.empty_like(activity, dtype=np.float64)

    low_mask = activity <= q_low
    voltage[low_mask] = V_ZERO + (V_ON - V_ZERO) * (
        activity[low_mask] / q_low
    )

    middle_mask = (activity > q_low) & (activity <= q_mid)
    voltage[middle_mask] = V_ON + (V_MID - V_ON) * (
        (activity[middle_mask] - q_low) / (q_mid - q_low)
    )

    high_mask = (activity > q_mid) & (activity < q_high)
    voltage[high_mask] = V_MID + (V_MAX - V_MID) * (
        (activity[high_mask] - q_mid) / (q_high - q_mid)
    )

    saturated_mask = activity >= q_high
    voltage[saturated_mask] = V_MAX
    np.clip(voltage, V_ZERO, V_MAX, out=voltage)

    if statistics is not None and prefix is not None:
        statistics[f"{prefix}_values"] += int(activity.size)
        statistics[f"{prefix}_low_band"] += int(np.count_nonzero(low_mask))
        statistics[f"{prefix}_main_band"] += int(np.count_nonzero(middle_mask))
        statistics[f"{prefix}_high_band"] += int(np.count_nonzero(high_mask))
        statistics[f"{prefix}_saturated"] += int(np.count_nonzero(saturated_mask))
    return voltage


def validate_mapping_anchors(mapping):
    q_low, q_mid, q_high = validate_mapping_parameters(mapping)
    probe = np.array([0.0, q_low, q_mid, q_high, q_high * 2.0])
    expected = np.array([V_ZERO, V_ON, V_MID, V_MAX, V_MAX])
    actual = map_activity_vin(probe, mapping)
    if not np.allclose(actual, expected, rtol=0.0, atol=1e-12):
        raise RuntimeError(f"电压锚点检查失败：{actual} != {expected}")
    dense_probe = np.linspace(0.0, q_high * 1.1, 10001)
    dense_voltage = map_activity_vin(dense_probe, mapping)
    if np.any(np.diff(dense_voltage) < -1e-12):
        raise RuntimeError("Vin 映射未保持单调不减。")


def map_payload_to_vin(data, source_name, mapping, statistics=None):
    values = data.numpy().astype(np.float64, copy=False)
    # 展平顺序固定为 stream、row、column，便于编码后无损恢复四维布局。
    streams_first = np.transpose(values, (1, 2, 3, 0)).reshape(-1, 1000)
    activity = streams_first
    voltage = map_activity_vin(
        activity,
        mapping,
        statistics=statistics,
        prefix=source_name,
    )
    if not np.all(np.isfinite(voltage)):
        raise ValueError(f"{source_name} 的 Vin 映射产生了 NaN 或无穷大。")
    # 第 1001 个 PWL 点重复末值，使 1000 个原始采样间隔完整覆盖 0～1 s。
    return np.ascontiguousarray(np.concatenate((voltage, voltage[:, -1:]), axis=1))


def build_output_payload(source_payload, spike_rows, streams):
    expected_runs = streams * 8 * 16
    if spike_rows.shape != (expected_runs, T):
        raise ValueError(
            f"spike_rows={spike_rows.shape}，期望 {(expected_runs, T)}。"
        )
    data = (
        torch.from_numpy(spike_rows.astype(np.bool_, copy=False))
        .reshape(streams, 8, 16, T)
        .permute(3, 0, 1, 2)
        .contiguous()
    )
    preserved = {
        key: value.clone() if isinstance(value, torch.Tensor) else value
        for key, value in source_payload.items()
        if key != "data"
    }
    return {"data": data, **preserved}


def metadata_equal(left, right):
    if isinstance(left, torch.Tensor) and isinstance(right, torch.Tensor):
        return torch.equal(left, right)
    return type(left) is type(right) and left == right


def validate_output_payload(source_payload, output_payload, path, streams):
    expected_shape = (T, streams, 8, 16)
    if not isinstance(output_payload, dict) or "data" not in output_payload:
        raise TypeError(f"{path} 必须是包含 data 的字典。")
    data = output_payload["data"]
    if not isinstance(data, torch.Tensor):
        raise TypeError(f"{path} 的 data 必须是 Tensor。")
    if tuple(data.shape) != expected_shape or data.dtype != torch.bool:
        raise ValueError(
            f"{path} data={tuple(data.shape)}/{data.dtype}，"
            f"期望 {expected_shape}/torch.bool。"
        )
    source_keys = set(source_payload) - {"data"}
    output_keys = set(output_payload) - {"data"}
    if source_keys != output_keys:
        raise ValueError(f"{path} 未完整保留源样本元数据。")
    for key in source_keys:
        if not metadata_equal(source_payload[key], output_payload[key]):
            raise ValueError(f"{path} 的元数据字段 {key} 与源样本不一致。")
    return data


def load_reusable_output(path, source_payload, streams):
    if OVERWRITE or not path.is_file():
        return None
    try:
        output_payload = load_pt(path)
        validate_output_payload(source_payload, output_payload, path, streams)
        return output_payload
    except (EOFError, KeyError, OSError, RuntimeError, TypeError, ValueError):
        return None


def binary_f1(reference, candidate):
    reference = np.asarray(reference, dtype=np.bool_)
    candidate = np.asarray(candidate, dtype=np.bool_)
    true_positive = int(np.count_nonzero(reference & candidate))
    denominator = int(reference.sum() + candidate.sum())
    return 1.0 if denominator == 0 else 2.0 * true_positive / denominator


def tolerant_f1(reference, candidate, tolerance=1):
    reference_bins = np.flatnonzero(reference)
    candidate_bins = np.flatnonzero(candidate)
    left = right = matches = 0
    while left < len(reference_bins) and right < len(candidate_bins):
        delta = candidate_bins[right] - reference_bins[left]
        if abs(delta) <= tolerance:
            matches += 1
            left += 1
            right += 1
        elif delta < -tolerance:
            right += 1
        else:
            left += 1
    denominator = len(reference_bins) + len(candidate_bins)
    return 1.0 if denominator == 0 else 2.0 * matches / denominator


def batched(items, batch_size):
    for start in range(0, len(items), batch_size):
        yield items[start:start + batch_size]

In [ ]:
source_files = {}
work_items = []
source_fingerprints = {}

for source_name, config in SOURCE_CONFIGS.items():
    source_files[source_name] = {}
    manifest_candidates = (
        config["root"] / "split_manifest.json",
        config["root"] / "manifest.json",
    )
    source_manifest = next(
        (path for path in manifest_candidates if path.is_file()),
        None,
    )
    if source_manifest is None:
        raise FileNotFoundError(
            f"{source_name} 未找到 split_manifest.json 或 manifest.json。"
        )
    source_fingerprints[source_name] = sha256sum(source_manifest)

    for split_name in SPLIT_NAMES:
        paths = sorted((config["root"] / split_name).glob("subject_*/*.pt"))
        if len(paths) != EXPECTED_COUNTS[split_name]:
            raise RuntimeError(
                f"{source_name}/{split_name} 样本数={len(paths)}，"
                f"期望 {EXPECTED_COUNTS[split_name]}。"
            )
        source_files[source_name][split_name] = paths

    for split_name in SPLIT_NAMES:
        for path in source_files[source_name][split_name]:
            work_items.append((source_name, split_name, path))

# 映射统计量来源：raw/train 的绝对幅值。
raw_train_paths = sorted(RAW_TRAIN_ROOT.glob("subject_*/*.pt"))
if len(raw_train_paths) != EXPECTED_COUNTS["train"]:
    raise RuntimeError(
        f"raw/train 样本数={len(raw_train_paths)}，期望 {EXPECTED_COUNTS['train']}。"
    )
if not RAW_MANIFEST_PATH.is_file():
    raise FileNotFoundError(f"缺少 raw 划分清单：{RAW_MANIFEST_PATH}")
raw_source_manifest_sha256 = sha256sum(RAW_MANIFEST_PATH)

print(f"待转换/检查样本：{len(work_items)}")
print("源数据数量与 manifest 检查通过。")

In [ ]:
def load_raw_training_values(path):
    payload = load_pt(path)
    data = validate_source_payload(payload, path, streams=1)
    return data.abs().reshape(-1)


def exact_linear_quantiles_from_histogram(values, probabilities, histogram_bins):
    # 先定位目标秩所在直方图区间，再只排序这些区间，避免对全量值整体排序。
    if values.ndim != 1 or values.numel() == 0:
        raise ValueError("分位统计输入必须是非空一维 Tensor。")
    value_count = values.numel()
    maximum = float(values.max().item())
    if not math.isfinite(maximum) or maximum <= 0.0:
        raise ValueError("raw/train 的绝对幅值最大值必须为有限正数。")

    bin_indices = torch.floor(
        values * (histogram_bins / maximum)
    ).to(torch.int64)
    bin_indices.clamp_(min=0, max=histogram_bins - 1)
    counts = torch.bincount(bin_indices, minlength=histogram_bins)
    cumulative = counts.cumsum(dim=0).cpu().numpy()

    rank_specs = []
    required_ranks = set()
    for probability in probabilities:
        rank = float(probability) * (value_count - 1)
        lower_rank = int(math.floor(rank))
        upper_rank = int(math.ceil(rank))
        rank_specs.append((lower_rank, upper_rank, rank - lower_rank))
        required_ranks.update((lower_rank, upper_rank))

    ranks_by_bin = {}
    for rank in sorted(required_ranks):
        bin_index = int(np.searchsorted(cumulative, rank, side="right"))
        count_before = 0 if bin_index == 0 else int(cumulative[bin_index - 1])
        ranks_by_bin.setdefault(bin_index, []).append((rank, rank - count_before))

    values_by_rank = {}
    for bin_index, targets in ranks_by_bin.items():
        candidates = torch.sort(values[bin_indices == bin_index]).values
        for rank, local_rank in targets:
            values_by_rank[rank] = float(candidates[local_rank].item())

    quantiles = []
    for lower_rank, upper_rank, weight in rank_specs:
        lower_value = values_by_rank[lower_rank]
        upper_value = values_by_rank[upper_rank]
        quantiles.append(lower_value + (upper_value - lower_value) * weight)
    return quantiles


def compute_global_vin_mapping(raw_train_paths):
    values_per_file = 1000 * 1 * 8 * 16
    total_values = len(raw_train_paths) * values_per_file
    use_pinned_memory = RESOLVED_QUANTILE_DEVICE == "cuda"
    training_abs = torch.empty(
        total_values,
        dtype=torch.float32,
        pin_memory=use_pinned_memory,
    )

    cursor = 0
    load_batch_size = max(IO_WORKERS * 2, 1)
    with ThreadPoolExecutor(max_workers=IO_WORKERS) as executor:
        with tqdm(
            total=len(raw_train_paths),
            unit="sample",
            desc="raw/train 全局分位统计",
        ) as progress:
            for path_batch in batched(raw_train_paths, load_batch_size):
                loaded = list(executor.map(load_raw_training_values, path_batch))
                for values in loaded:
                    next_cursor = cursor + values.numel()
                    training_abs[cursor:next_cursor].copy_(values)
                    cursor = next_cursor
                    progress.update(1)

    if cursor != total_values:
        raise RuntimeError(f"训练统计值数量={cursor}，期望 {total_values}。")

    with torch.inference_mode():
        if RESOLVED_QUANTILE_DEVICE == "cuda":
            training_abs_device = training_abs.to("cuda", non_blocking=True)
            torch.cuda.synchronize()
        else:
            training_abs_device = training_abs
        quantiles = exact_linear_quantiles_from_histogram(
            training_abs_device,
            (P_LOW, P_MID, P_HIGH),
            QUANTILE_HISTOGRAM_BINS,
        )

    q_low, q_mid, q_high = quantiles
    del training_abs_device, training_abs
    if RESOLVED_QUANTILE_DEVICE == "cuda":
        torch.cuda.empty_cache()

    mapping_core = {
        "version": "Vin",
        "quantile_method": "high_resolution_histogram_refined_exact_linear",
        "histogram_bins": QUANTILE_HISTOGRAM_BINS,
        "statistics_source": "CapgMyo_data/raw/train abs(raw)",
        "probabilities": {"low": P_LOW, "mid": P_MID, "high": P_HIGH},
        "voltage_anchors_v": {
            "zero": V_ZERO,
            "on": V_ON,
            "mid": V_MID,
            "max": V_MAX,
        },
        "q_low": q_low,
        "q_mid": q_mid,
        "q_high": q_high,
        "statistics_count": total_values,
        "raw_train_file_count": len(raw_train_paths),
        "raw_source_manifest_sha256": raw_source_manifest_sha256,
    }
    return {
        **mapping_core,
        "parameter_hash": canonical_hash(mapping_core),
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
    }


mapping_parameters = None
if MAPPING_PATH.is_file() and REUSE_MAPPING_PARAMETERS:
    candidate = json.loads(MAPPING_PATH.read_text(encoding="utf-8"))
    validate_mapping_parameters(candidate)
    expected_count = len(raw_train_paths) * 1000 * 8 * 16
    identity_matches = (
        candidate.get("version") == "Vin"
        and candidate.get("quantile_method")
        == "high_resolution_histogram_refined_exact_linear"
        and candidate.get("histogram_bins") == QUANTILE_HISTOGRAM_BINS
        and candidate.get("statistics_count") == expected_count
        and candidate.get("raw_train_file_count") == len(raw_train_paths)
        and candidate.get("raw_source_manifest_sha256") == raw_source_manifest_sha256
    )
    hash_payload = {
        key: value
        for key, value in candidate.items()
        if key not in {"parameter_hash", "created_at_utc"}
    }
    hash_matches = candidate.get("parameter_hash") == canonical_hash(hash_payload)
    if not identity_matches or not hash_matches:
        raise RuntimeError(
            "现有 mapping.json 与当前 raw/train 或参数不一致；"
            "请使用新的输出根目录，不允许静默重标定已有数据。"
        )
    mapping_parameters = candidate
    print("已复用现有 Vin 全局映射参数。")
else:
    if MAPPING_PATH.exists() and not OVERWRITE:
        raise RuntimeError(
            "mapping.json 已存在且 REUSE_MAPPING_PARAMETERS=False；"
            "如确认需要重算，请先使用新的输出根目录。"
        )
    mapping_parameters = compute_global_vin_mapping(raw_train_paths)
    validate_mapping_parameters(mapping_parameters)
    write_json_atomic(MAPPING_PATH, mapping_parameters)
    print("已计算并保存 Vin 全局映射参数。")

validate_mapping_anchors(mapping_parameters)
print(
    "Vin 分位点："
    f"q_low={mapping_parameters['q_low']:.9g}, "
    f"q_mid={mapping_parameters['q_mid']:.9g}, "
    f"q_high={mapping_parameters['q_high']:.9g}"
)
print(f"映射参数哈希：{mapping_parameters['parameter_hash']}")

expected_manifest_config = {
    "format_version": 3,
    "T": T,
    "duration_s": 1.0,
    "internal_step_s": INTERNAL_STEP_S,
    "mapping_version": "Vin",
    "mapping_parameter_hash": mapping_parameters["parameter_hash"],
    "mapping_parameter_path": MAPPING_PATH.relative_to(PROJECT_ROOT).as_posix(),
    "raw_polarity_activity": "positive/negative magnitudes with shared Vin parameters",
    "source_manifest_sha256": source_fingerprints,
}

if MANIFEST_PATH.is_file():
    existing_manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
    for key, value in expected_manifest_config.items():
        if existing_manifest.get(key) != value and not OVERWRITE:
            raise RuntimeError(
                f"现有 manifest 的 {key} 与本次参数不一致；"
                "请更换输出目录或确认后设置 OVERWRITE=True。"
            )

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

In [ ]:
preflight_vin = []
preflight_labels = []
for source_name in SOURCE_CONFIGS:
    split_name = SPLIT_NAMES[0]
    source_path = source_files[source_name][split_name][0]
    payload = load_pt(source_path)
    data = validate_source_payload(
        payload,
        source_path,
        SOURCE_CONFIGS[source_name]["streams"],
    )
    vin = map_payload_to_vin(data, source_name, mapping_parameters)
    indices = (
        list(range(REFERENCE_CHECK_CHANNELS))
        + list(range(128, 128 + REFERENCE_CHECK_CHANNELS))
    )
    labels = (
        [f"positive_ch{index:03d}" for index in range(REFERENCE_CHECK_CHANNELS)]
        + [f"negative_ch{index:03d}" for index in range(REFERENCE_CHECK_CHANNELS)]
    )
    preflight_vin.append(vin[indices])
    preflight_labels.extend(labels)

preflight_vin = np.concatenate(preflight_vin)
started = time.perf_counter()
try:
    preflight_fast = simulate_system_batch_spikes(
        preflight_vin,
        T=T,
        backend=RESOLVED_BACKEND,
        verbose=CUDA_BUILD_VERBOSE,
    )
except (OSError, RuntimeError) as error:
    if BACKEND == "auto" and RESOLVED_BACKEND == "cuda":
        print(f"警告：CUDA 后端初始化失败，自动回退到 NumPy：{error}")
        RESOLVED_BACKEND = "numpy"
        preflight_fast = simulate_system_batch_spikes(
            preflight_vin,
            T=T,
            backend="numpy",
        )
    else:
        raise

estimated_cuda_full_s = None
if RESOLVED_BACKEND == "cuda":
    # 使用足够多的重复真实流让 GPU 饱和；估算值不包含文件读写和首次编译。
    repeat_count = math.ceil(CUDA_BENCHMARK_STREAMS / len(preflight_vin))
    benchmark_vin = np.tile(preflight_vin, (repeat_count, 1))[:CUDA_BENCHMARK_STREAMS]
    benchmark_stream_count = len(benchmark_vin)
    torch.cuda.synchronize()
    benchmark_started = time.perf_counter()
    _ = simulate_system_batch_spikes(benchmark_vin, T=T, backend="cuda")
    torch.cuda.synchronize()
    benchmark_elapsed = time.perf_counter() - benchmark_started
    total_streams = sum(
        EXPECTED_COUNTS[split_name] * config["streams"] * 8 * 16
        for config in SOURCE_CONFIGS.values()
        for split_name in SPLIT_NAMES
    )
    estimated_cuda_full_s = benchmark_elapsed * total_streams / CUDA_BENCHMARK_STREAMS
    del benchmark_vin, _
    torch.cuda.empty_cache()
else:
    benchmark_elapsed = time.perf_counter() - started
    benchmark_stream_count = len(preflight_vin)

preflight_reference = simulate_system_batch_spikes_reference(
    preflight_vin,
    T=T,
    workers=1,
)
preflight_metrics = []
for label, reference, candidate in zip(
    preflight_labels,
    preflight_reference,
    preflight_fast,
):
    preflight_metrics.append({
        "run": label,
        "reference_spikes": int(reference.sum()),
        "candidate_spikes": int(candidate.sum()),
        "count_difference": int(candidate.sum()) - int(reference.sum()),
        "exact_f1": binary_f1(reference, candidate),
        "tolerant_f1_1bin": tolerant_f1(reference, candidate, tolerance=1),
    })

mean_exact_f1 = float(np.mean([row["exact_f1"] for row in preflight_metrics]))
minimum_tolerant_f1 = float(min(row["tolerant_f1_1bin"] for row in preflight_metrics))
maximum_count_difference = int(max(abs(row["count_difference"]) for row in preflight_metrics))
print(f"后端预热/基准批次耗时：{benchmark_elapsed:.3f} s / {benchmark_stream_count} streams")
if estimated_cuda_full_s is not None:
    print(
        f"按饱和批次粗略估算，纯 CUDA 求解约 {estimated_cuda_full_s / 60:.1f} 分钟；"
        "实际总耗时还包括映射、读写和输出复核。"
    )
print(f"平均精确 F1：{mean_exact_f1:.6f}")
print(f"最小 ±1 bin F1：{minimum_tolerant_f1:.6f}")
print(f"最大脉冲数差：{maximum_count_difference}")

if (
    mean_exact_f1 < MIN_EXACT_F1
    or minimum_tolerant_f1 < MIN_TOLERANT_F1
    or maximum_count_difference > MAX_SPIKE_COUNT_DIFFERENCE
):
    raise RuntimeError("批量后端与 Python 参考求解器的预检误差超限，已停止全量转换。")
print("后端预检通过。")

In [ ]:
batch_samples = CUDA_BATCH_SAMPLES if RESOLVED_BACKEND == "cuda" else CPU_BATCH_SAMPLES

conversion_stats = Counter()
mapping_statistics = Counter()
records = []
spike_statistics = {
    source_name: {
        split_name: {"spike_count": 0, "positions": 0, "samples": 0}
        for split_name in SPLIT_NAMES
    }
    for source_name in SOURCE_CONFIGS
}


def update_statistics(source_name, split_name, spikes):
    stats = spike_statistics[source_name][split_name]
    stats["spike_count"] += int(spikes.sum())
    stats["positions"] += int(spikes.numel())
    stats["samples"] += 1


def record_result(source_name, split_name, source_path, output_path, spikes, action):
    config = SOURCE_CONFIGS[source_name]
    update_statistics(source_name, split_name, spikes)
    records.append({
        "source": source_name,
        "split": split_name,
        "source_path": source_path.relative_to(PROJECT_ROOT).as_posix(),
        "output_path": output_path.relative_to(PROJECT_ROOT).as_posix(),
        "shape": list(spikes.shape),
        "dtype": str(spikes.dtype),
        "spike_count": int(spikes.sum()),
        "action": action,
        "streams": config["streams"],
    })


def load_conversion_entry(source_name, split_name, source_path):
    config = SOURCE_CONFIGS[source_name]
    relative_path = source_path.relative_to(config["root"] / split_name)
    output_path = OUTPUT_ROOT / source_name / split_name / relative_path
    source_payload = load_pt(source_path)
    validate_source_payload(
        source_payload,
        source_path,
        config["streams"],
    )
    reusable_output = load_reusable_output(
        output_path,
        source_payload,
        config["streams"],
    )
    return {
        "source": source_name,
        "split": split_name,
        "source_path": source_path,
        "output_path": output_path,
        "payload": source_payload,
        "streams": config["streams"],
        "reusable_output": reusable_output,
    }


def submit_load_batch(executor, source_name, split_name, paths):
    return [
        executor.submit(load_conversion_entry, source_name, split_name, path)
        for path in paths
    ]


def convert_batch(entries):
    vin_rows = []
    run_counts = []
    for entry in entries:
        data = entry["payload"]["data"]
        vin = map_payload_to_vin(
            data,
            entry["source"],
            mapping_parameters,
            statistics=mapping_statistics,
        )
        vin_rows.append(vin)
        run_counts.append(vin.shape[0])

    all_spikes = simulate_system_batch_spikes(
        np.concatenate(vin_rows),
        T=T,
        backend=RESOLVED_BACKEND,
    )
    cursor = 0
    for entry, run_count in zip(entries, run_counts):
        rows = all_spikes[cursor:cursor + run_count]
        cursor += run_count
        output_payload = build_output_payload(
            entry["payload"],
            rows,
            entry["streams"],
        )
        validate_output_payload(
            entry["payload"],
            output_payload,
            entry["output_path"],
            entry["streams"],
        )
        save_pt_atomic(entry["output_path"], output_payload)
        conversion_stats["written"] += 1
        record_result(
            entry["source"],
            entry["split"],
            entry["source_path"],
            entry["output_path"],
            output_payload["data"],
            "written",
        )


started = time.perf_counter()
with ThreadPoolExecutor(max_workers=IO_WORKERS) as loader:
    with tqdm(
        total=len(work_items),
        unit="sample",
        desc="System_with_TIA Encoding",
    ) as progress:
        for source_name in SOURCE_CONFIGS:
            for split_name in SPLIT_NAMES:
                selected_paths = [
                    path
                    for item_source, item_split, path in work_items
                    if item_source == source_name and item_split == split_name
                ]
                path_batches = list(batched(selected_paths, batch_samples))
                if not path_batches:
                    continue

                current_futures = submit_load_batch(
                    loader,
                    source_name,
                    split_name,
                    path_batches[0],
                )
                for batch_index, path_batch in enumerate(path_batches):
                    entries = [future.result() for future in current_futures]
                    if batch_index + 1 < len(path_batches):
                        next_futures = submit_load_batch(
                            loader,
                            source_name,
                            split_name,
                            path_batches[batch_index + 1],
                        )
                    else:
                        next_futures = None

                    pending = []
                    for entry in entries:
                        if entry["reusable_output"] is not None:
                            output_payload = entry["reusable_output"]
                            conversion_stats["reused"] += 1
                            record_result(
                                entry["source"],
                                entry["split"],
                                entry["source_path"],
                                entry["output_path"],
                                output_payload["data"],
                                "reused",
                            )
                        else:
                            pending.append(entry)

                    if pending:
                        convert_batch(pending)
                    progress.update(len(path_batch))
                    progress.set_postfix(
                        backend=RESOLVED_BACKEND,
                        source=source_name,
                        split=split_name,
                    )
                    current_futures = next_futures

conversion_elapsed_s = time.perf_counter() - started
print(f"转换阶段完成：{dict(conversion_stats)}，耗时 {conversion_elapsed_s:.1f} s")

In [ ]:
verification_counts = Counter()
with tqdm(total=len(work_items), unit="sample", desc="输出复核") as progress:
    for source_name, split_name, source_path in work_items:
        config = SOURCE_CONFIGS[source_name]
        relative_path = source_path.relative_to(config["root"] / split_name)
        output_path = OUTPUT_ROOT / source_name / split_name / relative_path
        source_payload = load_pt(source_path)
        output_payload = load_pt(output_path)
        validate_output_payload(
            source_payload,
            output_payload,
            output_path,
            config["streams"],
        )
        verification_counts[(source_name, split_name)] += 1
        progress.update(1)

for source_splits in spike_statistics.values():
    for stats in source_splits.values():
        stats["spike_rate"] = (
            stats["spike_count"] / stats["positions"]
            if stats["positions"]
            else None
        )

hardware = {
    "platform": platform.platform(),
    "python": platform.python_version(),
    "torch": torch.__version__,
    "cpu_count": os.cpu_count(),
    "torch_cpu_threads": torch.get_num_threads(),
    "io_workers": IO_WORKERS,
    "cuda_available": torch.cuda.is_available(),
}
if torch.cuda.is_available():
    hardware.update({
        "cuda_runtime": torch.version.cuda,
        "gpu": torch.cuda.get_device_name(torch.cuda.current_device()),
        "gpu_memory_bytes": torch.cuda.get_device_properties(0).total_memory,
    })

manifest = {
    **expected_manifest_config,
    "status": "complete",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "backend": RESOLVED_BACKEND,
    "quantile_device": RESOLVED_QUANTILE_DEVICE,
    "batch_samples": batch_samples,
    "conversion_elapsed_s": conversion_elapsed_s,
    "hardware": hardware,
    "mapping": mapping_parameters,
    "preflight": {
        "metrics": preflight_metrics,
        "mean_exact_f1": mean_exact_f1,
        "minimum_tolerant_f1_1bin": minimum_tolerant_f1,
        "maximum_spike_count_difference": maximum_count_difference,
        "benchmark_elapsed_s": benchmark_elapsed,
        "estimated_cuda_full_s": estimated_cuda_full_s,
    },
    "conversion_stats": dict(conversion_stats),
    "mapping_statistics_written_this_run": dict(mapping_statistics),
    "verification_counts": {
        f"{source}/{split_name}": count
        for (source, split_name), count in verification_counts.items()
    },
    "spike_statistics": spike_statistics,
    "records": records,
}
write_json_atomic(MANIFEST_PATH, manifest)

print("全量输出复核通过。")
print(f"manifest：{MANIFEST_PATH}")
for source_name, splits in spike_statistics.items():
    source_spikes = sum(item["spike_count"] for item in splits.values())
    source_positions = sum(item["positions"] for item in splits.values())
    rate = source_spikes / source_positions if source_positions else float("nan")
    print(f"{source_name} 脉冲占用率：{rate:.4%}")